In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [3]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [4]:
from dotenv import load_dotenv
from openai import OpenAI
from toyaikit.llm import OpenAIClient

load_dotenv()
openai_client = OpenAI()

In [5]:
def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [6]:
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [7]:
ground_truth[0]

{'question': 'I just found this course — is it too late to join now?',
 'document': '74eb249bbf'}

In [8]:
rec = ground_truth[0]

result = runner.loop(prompt=rec["question"])

In [9]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant. Answer student questions based on\nthe FAQ search results. Use the search tool before answering.", role='developer', phase=None, type=None),
 EasyInputMessage(content='I just found this course — is it too late to join now?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"too late to join course enrollment late add register after start FAQ"}', call_id='call_tMDDQmi1rslTLaNEMOo3hmLh', name='search', type='function_call', id='fc_0143a8fad6e8d121006a54d81a60e08199b6d0dbecaae35a89', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_tMDDQmi1rslTLaNEMOo3hmLh',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\

In [10]:
def extract_tool_calls(messages):
    tool_calls = []

    for message in messages:
        if isinstance(message, dict):
            continue

        if message.type == "function_call":
            tool_calls.append({
                "name": message.name,
                "arguments": message.arguments,
            })

    return tool_calls

In [11]:
tool_calls = extract_tool_calls(result.all_messages)

tool_calls

[{'name': 'search',
  'arguments': '{"query":"too late to join course enrollment late add register after start FAQ"}'}]

In [12]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

In [13]:
rec

{'question': 'I just found this course — is it too late to join now?',
 'document': '74eb249bbf'}

In [14]:
original_doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [15]:
answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [16]:
agent_result = {
    "question": rec["question"],
    "answer_agent": result.last_message,
    "answer_orig": answer_orig,
    "tool_calls": tool_calls,
    "cost": result.cost.total_cost,
    "document": doc_id,
}

agent_result

{'question': 'I just found this course — is it too late to join now?',
 'answer_agent': 'Yes — you can still join. The course materials are available anytime, and you can start whenever you want.\n\nOne important caveat: if you want a certificate, you need to submit the project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'tool_calls': [{'name': 'search',
   'arguments': '{"query":"too late to join course enrollment late add register after start FAQ"}'}],
 'cost': Decimal('0.00106575'),
 'document': '74eb249bbf'}

In [17]:
def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])

    tool_calls = extract_tool_calls(result.all_messages)

    answer_record = {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": tool_calls,
        "cost": result.cost.total_cost,
        "document": doc_id,
    }

    return answer_record

In [18]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    agent_answers = map_progress(pool, ground_truth[:50], generate_agent_answer)

  0%|          | 0/50 [00:00<?, ?it/s]

In [19]:
df_agent = pd.DataFrame(agent_answers)

In [20]:
df_agent.head()

,question,answer_agent,answer_orig,tool_calls,cost,document
0,I just found this course — is it too late to j...,Yes — you can still join anytime and start lea...,"Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""to...",0.00122175,74eb249bbf
1,Can I still start the course if I'm coming in ...,Yes — you can still start the course late.\n\n...,"Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""co...",0.001269,74eb249bbf
2,"If I enroll now, is there any chance to get a ...",Yes — but only if you can still finish the cou...,"Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""en...",0.00123975,74eb249bbf
3,What do I need to do to be eligible for the co...,To be eligible for the course certificate:\n\n...,"Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""co...",0.001446,74eb249bbf
4,Is the final project deadline the only thing t...,"No. For certification, the capstone project is...","Yes, but if you want to receive a certificate,...","[{'name': 'search', 'arguments': '{""query"":""fi...",0.00136275,74eb249bbf


In [21]:
df_agent["cost"].sum()

Decimal('0.06487125')

In [22]:
df_agent.to_csv("data/agent-answers.csv", index=False)

In [23]:
# !wget -O data/agent-answers.csv https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/04-evaluation/data/agent-answers.csv

In [24]:
# df_agent = pd.read_csv("data/agent-answers.csv")
agent_answers = df_agent.to_dict(orient="records")

In [25]:
df_agent.shape

(50, 6)

In [26]:
from pydantic import BaseModel, Field
from typing import Literal

class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )

In [27]:
agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.
""".strip()

agent_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()

In [28]:
import json
from evaluation_utils import calc_total_price, llm_structured_retry

def evaluate_agent_answer(rec, model="gpt-5.4-mini"):
    tool_calls = rec["tool_calls"]

    if isinstance(tool_calls, str):
        tool_calls = json.loads(tool_calls)

    prompt = agent_judge_prompt.format(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_agent=rec["answer_agent"],
        tool_calls=json.dumps(tool_calls, indent=2),
    )

    result, usage = llm_structured_retry(
        openai_client,
        agent_judge_instructions,
        prompt,
        AgentEvaluation,
        model=model,
    )

    return result, usage

In [29]:
agent_eval, usage = evaluate_agent_answer(agent_answers[0])

agent_eval

AgentEvaluation(answer_reasoning='The agent’s answer matches the ground truth. It correctly says you can still join, and it includes the key condition that to receive a certificate you must submit your project while submissions are still open. The extra details about materials being available are not in the original answer but do not conflict with it.', answer_score='good', trajectory_reasoning="The search query was relevant to the question and included important concepts like 'too late to join' and enrollment timing. Only one search was needed, and the tool use was reasonable for this simple FAQ-style question.", trajectory_score='good')

In [30]:
def judge_agent_record(rec):
    agent_eval, usage = evaluate_agent_answer(rec)

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_score": agent_eval.answer_score,
        "answer_reasoning": agent_eval.answer_reasoning,
        "trajectory_score": agent_eval.trajectory_score,
        "trajectory_reasoning": agent_eval.trajectory_reasoning,
    }

    return result, usage

In [32]:
agent_answers

[{'question': 'I just found this course — is it too late to join now?',
  'answer_agent': 'Yes — you can still join anytime and start learning on your own.\n\nA couple of things to note:\n- The videos and course materials are available, so you can begin whenever you want.\n- If you want a certificate, you need to submit your project while submissions are still open.\n\nIf you want, I can also point you to the course docs and platform so you can get started quickly.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'tool_calls': [{'name': 'search',
    'arguments': '{"query":"too late to join course late enrollment start anytime registration deadlines"}'}],
  'cost': Decimal('0.00122175'),
  'document': '74eb249bbf'},
 {'question': "Can I still start the course if I'm coming in late?",
  'answer_agent': 'Yes — you can still start the course late.\n\nThe FAQ says: **“You can start whenever you w

In [31]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, agent_answers, judge_agent_record)

  0%|          | 0/50 [00:00<?, ?it/s]

In [34]:
results

[({'question': 'I just found this course — is it too late to join now?',
   'document': '74eb249bbf',
   'answer_score': 'good',
   'answer_reasoning': 'The agent’s answer matches the ground truth. It says you can still join anytime, and it includes the key condition that to receive a certificate, you must submit your project while submissions are still open. This preserves the essential meaning of the original answer, though it adds extra detail about videos/materials and offers further help.',
   'trajectory_score': 'good',
   'trajectory_reasoning': "The search query is relevant to the user’s question about whether it is too late to join. It includes terms like 'too late to join', 'late enrollment', and 'deadlines', which are appropriate keywords. Only one search call was made, which is reasonable for this simple question, and the tool call supports the final answer."},
  ResponseUsage(input_tokens=527, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), 

In [33]:
agent_evaluations = []
usages = []

for evaluation, usage in results:
    agent_evaluations.append(evaluation)
    usages.append(usage)

In [35]:
df_agent_eval = pd.DataFrame(agent_evaluations)

In [38]:
df_agent_eval.head()

,question,document,answer_score,answer_reasoning,trajectory_score,trajectory_reasoning
0,I just found this course — is it too late to j...,74eb249bbf,good,The agent’s answer matches the ground truth. I...,good,The search query is relevant to the user’s que...
1,Can I still start the course if I'm coming in ...,74eb249bbf,good,The agent’s answer matches the ground truth. I...,good,"The tool call was relevant to the question, us..."
2,"If I enroll now, is there any chance to get a ...",74eb249bbf,good,The agent’s answer matches the ground truth. I...,good,The search query was relevant to the question ...
3,What do I need to do to be eligible for the co...,74eb249bbf,good,The agent’s answer contains the key requiremen...,good,The search query was relevant to the question ...
4,Is the final project deadline the only thing t...,74eb249bbf,bad,The agent answer does not match the ground tru...,good,The single search query was relevant to the qu...


In [36]:
calc_total_price(usages)

0.054048000000000006

In [37]:
df_agent_eval["answer_score"].value_counts()

answer_score
good    47
bad      3
Name: count, dtype: int64

In [39]:
df_agent_eval["trajectory_score"].value_counts()

trajectory_score
good    49
bad      1
Name: count, dtype: int64

In [40]:
df_agent_eval.to_csv("data/agent-evaluations.csv", index=False)